# 📈 Modelos Baseline e Avaliação de Métricas de Ranking

Este notebook implementa e avalia dois recomendadores baseline:
1. **Recomendador de Popularidade (PopularityRecommender)**: Recomenda os itens mais populares de forma global.
2. **Recomendador Aleatório (RandomRecommender)**: Sorteia itens aleatoriamente (serve como baseline inferior).

A avaliação é realizada no conjunto de teste usando as 4 métricas exigidas pelo Tech Challenge:
- **Precision@K**
- **Recall@K**
- **MAP@K** (Mean Average Precision)
- **NDCG@K** (Normalized Discounted Cumulative Gain)

--- 
### 1. Importações e Configurações

In [10]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Adicionar a raiz do projeto ao PYTHONPATH para permitir imports do recsys
project_root = (
    Path(__file__).resolve().parents[1]
    if "__file__" in locals()
    else Path(".").resolve().parent
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from recsys.utils.config import settings

np.random.seed(settings.random_seed)
print("Semente de reprodutibilidade fixada em:", settings.random_seed)
print("Raiz do projeto mapeada:", project_root)

Semente de reprodutibilidade fixada em: 42
Raiz do projeto mapeada: C:\Users\rapha\Desktop\Tech Challenge\Sistema de Recomendação\Tech-Challenge-02


### 2. Carregamento e Pré-processamento dos Dados
Caso o arquivo `rating.csv` não exista localmente, o notebook gera dados sintéticos para simulação do MovieLens 20M.

In [11]:
raw_data_path = settings.data_raw_dir / "rating.csv"

if raw_data_path.exists():
    print("📂 Carregando dataset original...")
    df = pd.read_csv(raw_data_path, nrows=settings.data_sample_size)
else:
    print("⚠️ rating.csv não encontrado. Gerando dados sintéticos...")
    n_interactions = 50_000
    df = pd.DataFrame(
        {
            "userId": np.random.randint(1, 1001, size=n_interactions),
            "movieId": np.random.randint(1, 501, size=n_interactions),
            "rating": np.random.choice(
                [1.0, 2.0, 3.0, 4.0, 5.0],
                size=n_interactions,
                p=[0.05, 0.1, 0.2, 0.35, 0.3],
            ),
            "timestamp": np.random.randint(1420070400, 1451606400, size=n_interactions),
        }
    )

# ── Binarização (Simulação de e-commerce) ──
df = df[df["rating"] >= 3.5].copy()
df["label"] = 1

# ── Label Encoding simplificado ──
from recsys.preprocessing.strategies import LabelEncodeStrategy

user_encoder = LabelEncodeStrategy()
item_encoder = LabelEncodeStrategy()

user_encoder.fit(df["userId"].tolist())
item_encoder.fit(df["movieId"].tolist())

df["user_id"] = user_encoder.transform(df["userId"].tolist())
df["item_id"] = item_encoder.transform(df["movieId"].tolist())

# ── Split Temporal (80% Treino / 20% Teste) ──
df = df.sort_values("timestamp")
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx][["user_id", "item_id", "label"]]
test_df = df.iloc[split_idx:][["user_id", "item_id", "label"]]

print(f"✅ Treino: {len(train_df):,} interações | Teste: {len(test_df):,} interações")
print(
    f"   Usuários: {user_encoder.vocabulary_size:,} | Itens: {item_encoder.vocabulary_size:,}"
)

📂 Carregando dataset original...
✅ Treino: 47,471 interações | Teste: 11,868 interações
   Usuários: 702 | Itens: 6,405


### 3. Implementação e Treinamento dos Baselines

In [12]:
# Tentar importar o PopularityRecommender do projeto, definindo um fallback se falhar
try:
    from recsys.models.popularity import PopularityRecommender

    print("✅ PopularityRecommender importado com sucesso do pacote recsys.")
except ImportError:
    print("⚠️ Falha no import do pacote. Usando definição local fallback...")

    class PopularityRecommender:
        def fit(self, user_ids, item_ids):
            # Contar popularidade de cada item
            counts = pd.Series(item_ids).value_counts()
            self._ranking = counts.index.tolist()

        def recommend(self, user_id, k=10):
            return self._ranking[:k]

✅ PopularityRecommender importado com sucesso do pacote recsys.


In [13]:
# Implementação do Recomendador Aleatório (RandomRecommender)
class RandomRecommender:
    def fit(self, user_ids, item_ids):
        self.all_items = list(set(item_ids))
        self.history = {}
        for u, i in zip(user_ids, item_ids):
            self.history.setdefault(u, set()).add(i)

    def recommend(self, user_id, k=10):
        seen = self.history.get(user_id, set())
        candidates = [item for item in self.all_items if item not in seen]
        if not candidates:
            return []
        return np.random.choice(
            candidates, size=min(k, len(candidates)), replace=False
        ).tolist()

In [14]:
# ── Treinar recomendadores baselines ──
train_users = train_df["user_id"].tolist()
train_items = train_df["item_id"].tolist()

pop_model = PopularityRecommender()
pop_model.fit(train_users, train_items)

rand_model = RandomRecommender()
rand_model.fit(train_users, train_items)

print("✅ Baselines ajustados (treinados) com sucesso!")

✅ Baselines ajustados (treinados) com sucesso!


### 4. Avaliação com Métricas de Ranking (Precision, Recall, MAP, NDCG)
Utilização das métricas implementadas no pacote `recsys.evaluation.metrics` para cálculo do desempenho no conjunto de teste.

In [15]:
# Tentar importar as métricas do projeto, definindo fallback se falhar
try:
    from recsys.evaluation.metrics import (
        map_at_k,
        ndcg_at_k,
        precision_at_k,
        recall_at_k,
    )

    print("✅ Métricas importadas com sucesso do pacote recsys.")
except ImportError:
    print("⚠️ Falha no import do pacote. Usando implementações de fallback...")

    def precision_at_k(actual, predicted, k=10):
        if k <= 0 or not predicted:
            return 0.0
        hits = len(set(predicted[:k]) & set(actual))
        return hits / k

    def recall_at_k(actual, predicted, k=10):
        if k <= 0 or not actual or not predicted:
            return 0.0
        hits = len(set(predicted[:k]) & set(actual))
        return hits / len(actual)

    def map_at_k(actuals, predictions, k=10):
        scores = []
        for act, pred in zip(actuals, predictions):
            if not act or not pred:
                continue
            score = 0.0
            num_hits = 0.0
            for i, p in enumerate(pred[:k]):
                if p in act:
                    num_hits += 1.0
                    score += num_hits / (i + 1.0)
            scores.append(score / min(len(act), k))
        return np.mean(scores) if scores else 0.0

    def ndcg_at_k(actual, predicted, k=10):
        if k <= 0 or not actual or not predicted:
            return 0.0
        pred_k = predicted[:k]
        dcg = sum(1.0 / np.log2(i + 2.0) for i, p in enumerate(pred_k) if p in actual)
        idcg = sum(1.0 / np.log2(i + 2.0) for i in range(min(len(actual), k)))
        return dcg / idcg if idcg > 0 else 0.0

✅ Métricas importadas com sucesso do pacote recsys.


In [16]:
# Montar Ground Truth do conjunto de teste e histórico de treino
ground_truth = {}
for uid, iid in zip(test_df["user_id"], test_df["item_id"]):
    ground_truth.setdefault(uid, set()).add(iid)

train_history = {}
for uid, iid in zip(train_df["user_id"], train_df["item_id"]):
    train_history.setdefault(uid, set()).add(iid)

# Avaliar apenas para usuários que estão tanto no treino quanto no teste
eval_users = [u for u in ground_truth if u in train_history]
print(f"👥 Avaliando para {len(eval_users):,} usuários no conjunto de teste...")

👥 Avaliando para 29 usuários no conjunto de teste...


In [17]:
# ── Gerar recomendações ──
K = 10
actuals = [ground_truth[u] for u in eval_users]

pop_preds = [pop_model.recommend(u, k=K) for u in eval_users]
rand_preds = [rand_model.recommend(u, k=K) for u in eval_users]

# ── Calcular Métricas para Popularidade ──
pop_metrics = {
    f"Precision@{K}": np.mean(
        [precision_at_k(act, pred, k=K) for act, pred in zip(actuals, pop_preds)]
    ),
    f"Recall@{K}": np.mean(
        [recall_at_k(act, pred, k=K) for act, pred in zip(actuals, pop_preds)]
    ),
    f"MAP@{K}": map_at_k(actuals, pop_preds, k=K),
    f"NDCG@{K}": np.mean(
        [ndcg_at_k(act, pred, k=K) for act, pred in zip(actuals, pop_preds)]
    ),
}

# ── Calcular Métricas para Aleatório ──
rand_metrics = {
    f"Precision@{K}": np.mean(
        [precision_at_k(act, pred, k=K) for act, pred in zip(actuals, rand_preds)]
    ),
    f"Recall@{K}": np.mean(
        [recall_at_k(act, pred, k=K) for act, pred in zip(actuals, rand_preds)]
    ),
    f"MAP@{K}": map_at_k(actuals, rand_preds, k=K),
    f"NDCG@{K}": np.mean(
        [ndcg_at_k(act, pred, k=K) for act, pred in zip(actuals, rand_preds)]
    ),
}

print("✅ Métricas calculadas com sucesso!")

✅ Métricas calculadas com sucesso!


### 5. Comparação e Discussão dos Resultados

In [18]:
results_df = pd.DataFrame(
    {
        "Popularidade (Baseline Forte)": pop_metrics,
        "Aleatório (Baseline Fraco)": rand_metrics,
    }
).T

results_df

,Precision@10,Recall@10,MAP@10,NDCG@10
Popularidade (Baseline Forte),0.055172,0.018131,0.023035,0.058243
Aleatório (Baseline Fraco),0.006897,0.000949,0.001067,0.005233


### Discussão:
- **Popularidade vs Aleatório**: Como esperado, recomendar os itens mais populares apresenta uma performance significativamente superior ao modelo aleatório. Isso ocorre porque o comportamento de navegação humana é fortemente influenciado pelo viés de popularidade (itens mais vistos/comprados).
- **Limitações do Baseline de Popularidade**: Embora forte, o baseline de popularidade oferece a **mesma** recomendação para todos os usuários do sistema, ignorando preferências e gostos individuais. O modelo baseado em Deep Learning (**NCF**) buscará resolver essa limitação personalizando as ofertas.